# MMA ranking system — beginner-friendly overview

This project builds *probabilistic* MMA rankings and GOAT metrics from two sources:

- **UFCStats**: detailed per-round stats for most UFC fights (Tier A).
- **Tapology**: broad multi-promotion bout outcomes and metadata (Tier C).

We use **multiple rating systems** (Elo, Glicko-2, dynamic latent skills) and optionally **combine them** into one final predictor and one final ranking.

---

## 1) Three-tier data reality (A/B/C)

### Tier A — full per-round stats
Use when a fight is in the mapped UFCStats subset and round rows exist:
- per-round totals (KD, sig strikes, TD, control, etc.)
- sig strikes by **target** and **position**

This tier powers:
- per-round performance scoring (**PS_round**)
- stronger rating updates

### Tier B — fight-level totals only
Use when UFCStats fight exists but round stats missing:
- fight totals only
- lower confidence than Tier A

### Tier C — outcome/method only
Most Tapology bouts are here:
- win/loss/draw/NC
- method (KO/TKO/SUB/DEC/etc.) and sometimes round/time
- many missing weight_class / promotion / method details

Tier C powers:
- broad coverage across promotions and sports
- conservative updates (since we don't see round detail)

---

## 2) What the systems do

### System A: Elo-PS
A fast, incremental rating:
- predicts win probability from rating difference
- updates ratings after each MMA fight
- can use PS (Tier A/B) to make updates more informative

### System B: Glicko-2-PS
Like Elo, but with uncertainty:
- each fighter has a rating + **rating deviation** (uncertainty)
- inactivity increases uncertainty automatically
- updates shrink when uncertainty is low

### System C: Dynamic factor cross-sport model
A time-varying latent skill model:
- fighter has a vector: **MMA**, **striking**, **grappling**, **durability**
- kickboxing updates striking more than grappling
- the next MMA fight uses the updated latent state as a prior

### System D: Stacked ensemble (optional, accuracy-max)
Combine multiple systems into one best-calibrated probability:
- learn weights via rolling-origin backtests
- output final probability `p_final`

### System E: Expected Win Rate (EWR) ranking
Convert probabilities into a stable leaderboard:
- expected win probability vs a reference pool
- useful when schedules are unbalanced

---

## 3) What you will serve (typical)

For each fighter and `as_of_date`:
- a rating mean (skill)
- an uncertainty (if available)
- optional factor scores (striking/grappling/durability)
- derived GOAT components (peak, longevity, strength-of-schedule, etc.)

All results are **as-of reproducible**: same inputs + same code version ⇒ same rating snapshot.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

ranking_math = import_module('elo_calculator.application.ranking.math_utils')
logistic = ranking_math.logistic

for rating_diff in [-400, -200, 0, 200, 400]:
    probability = logistic(rating_diff / 400.0)
    print(f'rating_diff={rating_diff:>4} -> win_prob={probability:.3f}')

## 4) How to read the other notebooks

- **PS_round notebook**: defines per-round scoring aligned to damage > dominance > duration.
- **System A/B notebooks**: show how ratings update with toy data and how Tier A/B/C affects updates.
- **System C notebook**: shows cross-sport transfer with a simple example.
- **System D/E notebooks**: show how to combine systems and how to rank from probabilities.

If you only want one system:
- Choose **System C** for “most comprehensive” (cross-sport + time dynamics).
If you want the best predictive accuracy:
- Use **System C + stacking** (System D) and rank via **EWR** (System E).
